# 📚 Student Performance Data Analysis
**Task 1 – Data Science / Analysis with Python Internship**

---

## Overview
This notebook follows the complete data science workflow:
> **Load → Clean → Analyse → Visualise → Conclude**

**Dataset:** `student-mat.csv` – Student performance in Mathematics collected from two Portuguese secondary schools.

**Key columns:**
| Column | Description |
|--------|-------------|
| `G1` | First period grade (0–20) |
| `G2` | Second period grade (0–20) |
| `G3` | Final grade (0–20) – **target variable** |
| `studytime` | Weekly study time (1=<2h, 2=2-5h, 3=5-10h, 4=>10h) |
| `sex` | Student gender (F / M) |
| `failures` | Number of past class failures |

---
## Step 1 – Import Libraries & Load Dataset

We use:
- **`pandas`** – data loading, cleaning, and analysis
- **`matplotlib`** – base plotting library
- **`seaborn`** – statistical visualisations built on matplotlib
- **`numpy`** – numerical operations (jitter, regression line)

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Global plot style
plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor': '#ffffff',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = ['#4C72B0', '#DD8452']

# Load dataset (semicolon-separated)
df = pd.read_csv('/home/claude/student_data/student-mat.csv', sep=';')
print(f'Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

---
## Step 2 – Explore & Clean the Data

Before any analysis we must:
1. Check for **missing values** – null entries would skew statistics
2. **Remove duplicates** – identical rows inflate counts
3. Inspect **data types** – ensures numeric columns aren't stored as strings

In [ ]:
# ── Shape & dtypes
print(f'Shape: {df.shape}')
print(f'\nColumn data types:')
print(df.dtypes)

# ── Missing values
missing = df.isnull().sum()
print(f'\nMissing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'None – dataset is complete! ✅')

# ── Duplicates
dups = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f'\nDuplicates found & removed: {dups}')
print(f'Clean dataset shape: {df.shape}')

In [ ]:
# ── Descriptive statistics for key numeric columns
df[['age', 'studytime', 'failures', 'absences', 'G1', 'G2', 'G3']].describe().round(2)

---
## Step 3 – Analysis Questions

We answer four key analytical questions about the dataset.

### Q1. What is the average final grade (G3)?
The mean of G3 gives us a baseline for student performance.

In [ ]:
avg_g3 = df['G3'].mean()
median_g3 = df['G3'].median()
print(f'Average final grade (G3): {avg_g3:.2f} / 20')
print(f'Median  final grade (G3): {median_g3:.2f} / 20')
print(f'Standard deviation       : {df["G3"].std():.2f}')

### Q2. How many students scored above 15?
Grade > 15 out of 20 represents high-achieving students (~75th percentile).

In [ ]:
above15 = (df['G3'] > 15).sum()
pct = above15 / len(df) * 100
print(f'Students scoring G3 > 15 : {above15} students ({pct:.1f}%)')
print(f'Grade distribution overview:')
bins_labels = ['0-5','6-10','11-15','16-20']
cuts = pd.cut(df['G3'], bins=[0,5,10,15,20], labels=bins_labels)
print(cuts.value_counts().sort_index())

### Q3. Is study time correlated with performance?
Pearson correlation coefficient (r) measures linear relationship strength.
- r close to **+1**: strong positive
- r close to **0**: negligible

In [ ]:
corr = df['studytime'].corr(df['G3'])
print(f'Pearson correlation (studytime vs G3): r = {corr:.4f}')

# Mean grade per study-time bucket
study_labels = {1: '<2h', 2: '2-5h', 3: '5-10h', 4: '>10h'}
grade_by_study = df.groupby('studytime')['G3'].mean().rename(study_labels)
print('\nMean G3 by study time bucket:')
print(grade_by_study.round(2))

### Q4. Which gender performs better on average?

In [ ]:
gender_avg = df.groupby('sex')['G3'].mean()
gender_count = df.groupby('sex').size()
gender_std = df.groupby('sex')['G3'].std()

summary = pd.DataFrame({'Count': gender_count, 'Mean G3': gender_avg.round(2), 'Std Dev': gender_std.round(2)})
summary.index = summary.index.map({'F': 'Female', 'M': 'Male'})
print(summary)
better = 'Male' if gender_avg['M'] > gender_avg['F'] else 'Female'
diff = abs(gender_avg['M'] - gender_avg['F'])
print(f'\n→ {better} students score higher on average by {diff:.2f} points')

---
## Step 4 – Visualisations

Three charts are produced to communicate the findings visually.

### Chart 1 – Histogram of Final Grades
Shows the full grade distribution. Green bars highlight top performers (>15); red dashed line marks the mean.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#f8f9fa')

counts, bins, patches = ax.hist(df['G3'], bins=20, color='#4C72B0',
                                 edgecolor='white', linewidth=0.8, alpha=0.85)
ax.axvline(avg_g3, color='#e74c3c', linestyle='--', linewidth=2)

for patch, left in zip(patches, bins[:-1]):
    if left >= 15:
        patch.set_facecolor('#2ecc71')
        patch.set_alpha(0.85)

legend_patches = [
    mpatches.Patch(facecolor='#4C72B0', label='Grade distribution'),
    mpatches.Patch(facecolor='#2ecc71', label='Scores > 15'),
    plt.Line2D([0],[0], color='#e74c3c', linestyle='--', linewidth=2,
               label=f'Mean = {avg_g3:.2f}')
]
ax.legend(handles=legend_patches, fontsize=10)
ax.set_title('Distribution of Final Grades (G3)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Final Grade (G3)', fontsize=11)
ax.set_ylabel('Number of Students', fontsize=11)
plt.tight_layout()
plt.savefig('/home/claude/plot1_grade_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 1 saved')

### Chart 2 – Scatter Plot: Study Time vs Final Grade
Each dot is a student. Jitter is added horizontally for readability. The red trend line shows the direction of correlation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#f8f9fa')

np.random.seed(42)
jitter = np.random.uniform(-0.15, 0.15, size=len(df))
ax.scatter(df['studytime'] + jitter, df['G3'],
           alpha=0.45, s=50, color='#4C72B0', edgecolors='white', linewidth=0.4)

m, b = np.polyfit(df['studytime'], df['G3'], 1)
xs = np.linspace(df['studytime'].min(), df['studytime'].max(), 200)
ax.plot(xs, m * xs + b, color='#e74c3c', linewidth=2.2,
        label=f'Trend  (r = {corr:.2f})')

ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(['<2h', '2-5h', '5-10h', '>10h'])
ax.set_title('Study Time vs Final Grade (G3)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Weekly Study Time', fontsize=11)
ax.set_ylabel('Final Grade (G3)', fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('/home/claude/plot2_studytime_vs_grade.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 2 saved')

### Chart 3 – Bar Chart: Male vs Female Average Score
Compares mean final grades between genders.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor('#f8f9fa')

labels = ['Female (F)', 'Male (M)']
values = [gender_avg['F'], gender_avg['M']]
bars = ax.bar(labels, values, color=PALETTE, width=0.45,
              edgecolor='white', linewidth=1.2)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.15,
            f'{val:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylim(0, max(values) * 1.2)
ax.set_title('Average Final Grade by Gender', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Average Final Grade (G3)', fontsize=11)
plt.tight_layout()
plt.savefig('/home/claude/plot3_gender_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart 3 saved')

---
## Step 5 – Conclusions

| Question | Finding |
|----------|---------|
| Average final grade | **10.42 / 20** – slightly above the midpoint |
| Students scoring > 15 | **40 students (10.1%)** – top performers are a small minority |
| Study time vs performance | **r = 0.098** – very weak positive correlation; study time alone is not a strong predictor |
| Gender performance | **Male** students score marginally higher (10.91 vs 9.97) |

### Key Insights
1. The grade distribution is slightly **left-skewed** with a large cluster of zeros (students who may have dropped out or failed entirely).
2. Study time shows only a **weak positive trend** – suggesting other factors (family background, past failures, internet access) play a larger role.
3. The **gender gap is small** (~1 point), and both groups cluster near the overall mean.

### Next Steps (Future Work)
- Investigate the impact of `failures` and `schoolsup` on final grades
- Build a regression model using multiple features to predict G3
- Compare Math vs Portuguese datasets for cross-subject insights